# 🩺 Smart Diabetes Monitoring and Prediction System
## Notebook 3 — Baseline Models (Phase 1)

Trains and evaluates all 7 algorithms on the original 21 BRFSS features. Saves results for Notebook 5.

### Setup

In [1]:
import os, sys, time, warnings, pickle, json
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef, balanced_accuracy_score, log_loss
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from scipy.stats import pointbiserialr

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True
    print("✓ imbalanced-learn available")
except ImportError:
    SMOTE_AVAILABLE = False
    print("✗ imbalanced-learn not installed")

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print("✓ XGBoost available")
except ImportError:
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
    print("✓ LightGBM available")
except ImportError:
    LGB_AVAILABLE = False

try:
    import catboost as cb
    CAT_AVAILABLE = True
    print("✓ CatBoost available")
except ImportError:
    CAT_AVAILABLE = False

print("\n✅ All libraries loaded.")

✓ imbalanced-learn available
✓ XGBoost available
✓ LightGBM available
✓ CatBoost available

✅ All libraries loaded.


In [ ]:
# ── UPDATE THIS PATH TO YOUR DATASET LOCATION ────────────────────────
DATA_PATH   = r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Dataset\diabetes_binary_health_indicators_BRFSS2015.csv"
OUTPUT_ROOT = Path(r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Outputs_Folder")

DIRS = {
    "models"       : OUTPUT_ROOT / "models",
    "scalers"      : OUTPUT_ROOT / "scalers",
    "plots_eda"    : OUTPUT_ROOT / "plots" / "01_eda",
    "plots_imb"    : OUTPUT_ROOT / "plots" / "02_imbalance",
    "plots_pre"    : OUTPUT_ROOT / "plots" / "03_preprocessing",
    "plots_base"   : OUTPUT_ROOT / "plots" / "04_baseline_models",
    "plots_eng"    : OUTPUT_ROOT / "plots" / "05_feature_engineering",
    "plots_eng_m"  : OUTPUT_ROOT / "plots" / "06_engineered_models",
    "plots_fi"     : OUTPUT_ROOT / "plots" / "07_feature_importance",
    "plots_sel"    : OUTPUT_ROOT / "plots" / "08_selected_models",
    "plots_cmp"    : OUTPUT_ROOT / "plots" / "09_model_comparison",
    "plots_best"   : OUTPUT_ROOT / "plots" / "10_best_model",
    "reports"      : OUTPUT_ROOT / "reports",
    "features"     : OUTPUT_ROOT / "feature_lists",
    "thresholds"   : OUTPUT_ROOT / "thresholds",
    "pipeline"     : OUTPUT_ROOT / "pipeline",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Output root: {OUTPUT_ROOT.resolve()}")

✅ Output root: C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder


In [3]:
PALETTE   = ["#00B4D8","#0077B6","#48CAE4","#90E0EF",
             "#ADE8F4","#023E8A","#03045E","#CAF0F8"]
COLOR_POS = "#EF233C"
COLOR_NEG = "#00B4D8"
COLOR_ACC = "#2EC4B6"
COLOR_AUC = "#FF9F1C"
COLOR_ROC = "#0077B6"
COLOR_PR  = "#2EC4B6"
COLOR_F1  = "#FF9F1C"

sns.set_style("whitegrid")
sns.set_palette(PALETTE)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor"  : "#F8FBFF",
    "axes.edgecolor"  : "#CCCCCC",
    "axes.titlesize"  : 14,
    "axes.labelsize"  : 12,
    "xtick.labelsize" : 10,
    "ytick.labelsize" : 10,
    "font.family"     : "DejaVu Sans",
})

SEED     = 42
CV_FOLDS = 5
TOP_N    = 15
np.random.seed(SEED)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

all_results = {}

print(f"✅ SEED={SEED}  CV_FOLDS={CV_FOLDS}  TOP_N={TOP_N}")

✅ SEED=42  CV_FOLDS=5  TOP_N=15


In [4]:
def save_pkl(obj, path, label=""):
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  [SAVED] {label} → {path}")

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def savefig(fig, path, dpi=150):
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [PLOT]  {path.name}")

def section(title):
    print("\n" + "━"*70)
    print(f"  {title}")
    print("━"*70)

def print_classification_report(y_true, y_pred, model_name, phase=""):
    print(f"\n{'─'*60}")
    print(f"  📊 Classification Report — {model_name}")
    if phase:
        print(f"     Phase: {phase}")
    print(f"{'─'*60}")
    print(classification_report(
        y_true, y_pred,
        target_names=["No Diabetes (0)", "Diabetes (1)"],
        digits=4))
    print(f"{'─'*60}")

def evaluate_model(model, X_tr, y_tr, X_te, y_te, name,
                   cv_obj=None, phase="", verbose=True):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred  = model.predict(X_te)
    y_proba = (model.predict_proba(X_te)[:, 1]
               if hasattr(model, "predict_proba") else None)
    metrics = dict(
        accuracy      = accuracy_score(y_te, y_pred),
        precision     = precision_score(y_te, y_pred, zero_division=0),
        recall        = recall_score(y_te, y_pred, zero_division=0),
        f1            = f1_score(y_te, y_pred, zero_division=0),
        balanced_acc  = balanced_accuracy_score(y_te, y_pred),
        mcc           = matthews_corrcoef(y_te, y_pred),
        roc_auc       = roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
        avg_precision = average_precision_score(y_te, y_proba) if y_proba is not None else np.nan,
        log_loss_val  = log_loss(y_te, y_proba) if y_proba is not None else np.nan,
        train_time_s  = elapsed,
        cv_roc_auc    = np.nan,
    )
    if cv_obj is not None:
        scores = cross_val_score(model, X_tr, y_tr,
                                 cv=cv_obj, scoring="roc_auc", n_jobs=-1)
        metrics["cv_roc_auc"] = scores.mean()
    if verbose:
        print(f"  {name:<35} Acc={metrics['accuracy']:.4f}  "
              f"F1={metrics['f1']:.4f}  Recall={metrics['recall']:.4f}  "
              f"Prec={metrics['precision']:.4f}  AUC={metrics['roc_auc']:.4f}  "
              f"t={elapsed:.1f}s")
        print_classification_report(y_te, y_pred, name, phase)
    return metrics

def plot_roc_pr(y_true, models_proba, title_prefix, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, proba in models_proba.items():
        fpr, tpr, _ = roc_curve(y_true, proba)
        auc = roc_auc_score(y_true, proba)
        axes[0].plot(fpr, tpr, lw=2, label=f"{name}  AUC={auc:.3f}")
        prec, rec, _ = precision_recall_curve(y_true, proba)
        ap = average_precision_score(y_true, proba)
        axes[1].plot(rec, prec, lw=2, label=f"{name}  AP={ap:.3f}")
    axes[0].plot([0,1],[0,1],"k--",lw=1,alpha=.5)
    axes[0].set(xlabel="FPR", ylabel="TPR", title=f"{title_prefix} — ROC Curve")
    axes[0].legend(fontsize=7)
    baseline_ap = y_true.mean()
    axes[1].axhline(baseline_ap, color="gray", linestyle="--",
                    label=f"Baseline AP={baseline_ap:.3f}")
    axes[1].set(xlabel="Recall", ylabel="Precision",
                title=f"{title_prefix} — Precision-Recall Curve")
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    savefig(fig, save_dir / f"{title_prefix.replace(' ','_')}_roc_pr.png")
    plt.show()

def results_to_df(results_dict):
    rows = []
    for phase, models in results_dict.items():
        for mname, m in models.items():
            row = {"Phase": phase, "Model": mname}
            row.update(m)
            rows.append(row)
    return pd.DataFrame(rows)

def get_models():
    m = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, random_state=SEED, class_weight="balanced"),
        "Random Forest"      : RandomForestClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Extra Trees"        : ExtraTreesClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Gradient Boosting"  : GradientBoostingClassifier(
            n_estimators=200, random_state=SEED),
        "AdaBoost"           : AdaBoostClassifier(
            n_estimators=100, random_state=SEED),
    }
    if XGB_AVAILABLE:
        m["XGBoost"] = xgb.XGBClassifier(
            n_estimators=200, use_label_encoder=False,
            eval_metric="logloss", random_state=SEED, n_jobs=-1,
            scale_pos_weight=6)
    if LGB_AVAILABLE:
        m["LightGBM"] = lgb.LGBMClassifier(
            n_estimators=200, random_state=SEED, n_jobs=-1,
            class_weight="balanced", verbose=-1,
            scale_pos_weight=6,
            min_child_samples=5,
            min_split_gain=0.0)
    if CAT_AVAILABLE:
        m["CatBoost"] = cb.CatBoostClassifier(
            iterations=200, random_state=SEED, verbose=0,
            auto_class_weights="Balanced")
    return m

print("✅ All helper functions defined.")

✅ All helper functions defined.


### Load Outputs from Notebook 2

In [5]:
TARGET            = "Diabetes_binary"
FEATURES_ORIGINAL = load_pkl(DIRS["reports"] / "features_original.pkl")
X_train_scaled    = load_pkl(DIRS["reports"] / "X_train_scaled.pkl")
X_test_scaled     = load_pkl(DIRS["reports"] / "X_test_scaled.pkl")
y_train           = load_pkl(DIRS["reports"] / "y_train.pkl")
y_test            = load_pkl(DIRS["reports"] / "y_test.pkl")

print(f"  X_train : {X_train_scaled.shape}")
print(f"  X_test  : {X_test_scaled.shape}")
print("✅ Notebook 2 outputs loaded.")

  X_train : (183579, 21)
  X_test  : (45895, 21)
✅ Notebook 2 outputs loaded.


### Step 8 — Baseline Models on Original Features

In [6]:
section("STEP 8 — BASELINE MODELS (ORIGINAL FEATURES)")
print("  All models use class_weight=balanced or scale_pos_weight=6 depending on algorithm\n")

baseline_results, baseline_proba, baseline_trained = {}, {}, {}

for name, model in get_models().items():
    metrics = evaluate_model(
        model, X_train_scaled, y_train,
        X_test_scaled, y_test, name,
        cv_obj=cv, phase="01_Baseline")
    baseline_results[name] = metrics
    baseline_trained[name] = model
    if hasattr(model, "predict_proba"):
        baseline_proba[name] = model.predict_proba(X_test_scaled)[:, 1]

all_results["01_Baseline"] = baseline_results

for name, model in baseline_trained.items():
    save_pkl(model, DIRS["models"] / f"baseline_{name.replace(' ','_')}.pkl", name)
save_pkl(baseline_results,  DIRS["reports"] / "baseline_results.pkl",  "Baseline results")
save_pkl(baseline_trained,  DIRS["reports"] / "baseline_trained.pkl",  "Baseline trained")
save_pkl(baseline_proba,    DIRS["reports"] / "baseline_proba.pkl",    "Baseline proba")
print("\n✅ Baseline training complete.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 8 — BASELINE MODELS (ORIGINAL FEATURES)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  All models use class_weight=balanced or scale_pos_weight=6 depending on algorithm

  Logistic Regression                 Acc=0.7143  F1=0.4485  Recall=0.7595  Prec=0.3182  AUC=0.8106  t=0.9s

────────────────────────────────────────────────────────────
  📊 Classification Report — Logistic Regression
     Phase: 01_Baseline
────────────────────────────────────────────────────────────
                 precision    recall  f1-score   support

No Diabetes (0)     0.9421    0.7062    0.8073     38876
   Diabetes (1)     0.3182    0.7595    0.4485      7019

       accuracy                         0.7143     45895
      macro avg     0.6301    0.7329    0.6279     45895
   weighted avg     0.8467    0.7143    0.7524     45895

────────────────────────────────────────────────────────────
  Random Forest 

In [7]:
# Performance Bar Chart
bdf = pd.DataFrame(baseline_results).T[
    ["accuracy","precision","recall","f1","roc_auc","balanced_acc"]]

fig, ax = plt.subplots(figsize=(15, 6))
x = np.arange(len(bdf)); w = 0.13
metric_palette = [COLOR_ACC, PALETTE[1], COLOR_POS, COLOR_AUC, PALETTE[5], "#9B5DE5"]
for i, (col, color) in enumerate(zip(bdf.columns, metric_palette)):
    ax.bar(x + i*w, bdf[col], width=w,
           label=col.replace("_"," ").title(),
           color=color, edgecolor="white", linewidth=0.8)
ax.set_xticks(x + w * 2.5)
ax.set_xticklabels(bdf.index, rotation=30, ha="right")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9, ncol=3); ax.grid(axis="y", alpha=0.4)
ax.set_title("Baseline Models — Performance on UNBALANCED Data",
             fontweight="bold", fontsize=14)
ax.set_ylabel("Score")
fig.tight_layout()
savefig(fig, DIRS["plots_base"] / "01_baseline_comparison.png")
plt.show()

  [PLOT]  01_baseline_comparison.png


In [8]:
# ROC and PR Curves
plot_roc_pr(y_test, baseline_proba, "Baseline Models", DIRS["plots_base"])

  [PLOT]  Baseline_Models_roc_pr.png


In [9]:
# Confusion Matrices
n_models = len(baseline_trained)
ncols_cm, nrows_cm = 3, (n_models + 2) // 3
fig, axes = plt.subplots(nrows_cm, ncols_cm,
                          figsize=(ncols_cm * 5, nrows_cm * 4.5))
axes = axes.flatten()
for i, (name, model) in enumerate(baseline_trained.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No Diab", "Diab"],
                yticklabels=["No Diab", "Diab"],
                ax=axes[i], linewidths=1, linecolor="white",
                annot_kws={"size": 12})
    r = recall_score(y_test, y_pred, zero_division=0)
    p = precision_score(y_test, y_pred, zero_division=0)
    axes[i].set_title(f"{name}\nRecall={r:.3f}  Precision={p:.3f}",
                       fontweight="bold", fontsize=9)
for j in range(i + 1, len(axes)): axes[j].set_visible(False)
fig.suptitle("Baseline Models — Confusion Matrices",
             fontsize=14, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_base"] / "02_confusion_matrices.png")
plt.show()

  [PLOT]  02_confusion_matrices.png


In [10]:
# Summary Table
bdf_disp = pd.DataFrame(baseline_results).T[[
    "accuracy","precision","recall","f1",
    "roc_auc","avg_precision","balanced_acc","mcc","cv_roc_auc"]]
bdf_disp.columns = ["Accuracy","Precision","Recall","F1",
                     "ROC-AUC","PR-AUC","Bal.Acc","MCC","CV ROC-AUC"]
bdf_disp.round(4).sort_values("ROC-AUC", ascending=False)

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Bal.Acc,MCC,CV ROC-AUC
Gradient Boosting,0.8557,0.5944,0.1777,0.2736,0.8193,0.4473,0.5779,0.2684,0.8153
AdaBoost,0.8529,0.5545,0.1943,0.2878,0.8142,0.4336,0.5831,0.2655,0.8091
LightGBM,0.4336,0.2086,0.9681,0.3433,0.8138,0.4412,0.6526,0.2420,0.8092
CatBoost,0.7150,0.3206,0.7716,0.4530,0.8131,0.4338,0.7382,0.3556,0.8078
Logistic Regression,0.7143,0.3182,0.7595,0.4485,0.8106,0.4121,0.7329,0.3482,0.8067
XGBoost,0.7067,0.3131,0.7686,0.4449,0.8051,0.4204,0.7321,0.3450,0.8006
Random Forest,0.8443,0.4713,0.1507,0.2284,0.7814,0.3580,0.5601,0.2006,0.7806
Extra Trees,0.8330,0.3909,0.1647,0.2318,0.7512,0.3143,0.5592,0.1735,0.7533


### ✅ Notebook 3 Complete
Baseline results saved. Run Notebook 4 next.